# Search complexity and agreement in multi-issue negotiations

This notebook reproduces Study 1 from the included coded data. It does not use
papers, teaching materials, an API key or an LLM. Rebuilding the coding from
those source documents is a separate task, explained in DATASET.md.

One observation links a specific payoff structure and reservation rule to an
observed agreement rate. A scenario name can appear more than once when its
structure changes. The main sample includes documented reconstructions; the
stricter sample removes them. The source file retains every coded record,
including the record held out after the November audit.

Run all cells in order. The default is 5,000 draws for every resampling check.
All results are written to outputs/.

In [ ]:
from pathlib import Path
import json
import platform
import importlib.metadata
import pandas as pd

BASE = Path.cwd()
if not (BASE/'data/search_complexity_dataset.csv').exists():
    raise FileNotFoundError('Run this notebook from the repository folder.')
OUTPUT = BASE/'outputs'
OUTPUT.mkdir(exist_ok=True)
DRAWS = 5000
data = pd.read_csv(BASE/'data/search_complexity_dataset.csv')
versions = {name:importlib.metadata.version(name) for name in
            ['numpy','pandas','scipy','statsmodels','matplotlib']}
versions['python'] = platform.python_version()
(OUTPUT/'software_versions.json').write_text(json.dumps(versions,indent=2),encoding='utf-8')
print(versions)

## What we test

- **H1:** Bargaining-zone share is positively associated with agreement (Model 4).
- **H2:** Search complexity is negatively associated with agreement (Model 1).
- **H3:** Search complexity explains more variance than bargaining-zone share,
  with the same basic controls (R² of Model 2 minus R² of Model 5).
- **H4:** Search complexity remains negatively associated with agreement after
  bargaining-zone share and basic controls are included (Model 3).

The five controls are issues, parties, compatible issues, automated counterpart,
and a non-unanimous decision rule. Model 6 repeats Model 3, weighting each row
by the number of negotiation encounters supporting its outcome. Participants
are never substituted for negotiation encounters. A mean of independent study
rates remains the primary outcome even when pooled counts are available.

Continuous X variables are standardized after choosing a model's rows. Y stays
a proportion. A coefficient of −.10 therefore means 10 percentage points less
agreement per one-SD increase in X. We report **b**, rather than calling this
a fully standardized beta. Binary and fractional composition controls retain
their coding.

## Reusable analysis functions

The functions below fit the specified models with statsmodels and HC3 robust
standard errors. Tests are two-sided and use the t distribution. The source data
are checked before fitting. We do not statistically impute missing values.

One detail needs explicit handling. The strict sample has no automated
counterparts, and its non-unanimous-rule indicator occurs in one row. That dummy
fits its row exactly, making the ordinary HC3 expression 0/0. The code absorbs
the singleton dummy when calculating uncertainty for the remaining coefficients.
It verifies that their point estimates do not change. The dummy coefficient
itself has no HC3 standard error. This is not handled by silently clipping a
zero denominator.

For fast resampling we use the equivalent NumPy least-squares calculation.
The next check compares its point estimates with statsmodels.

In [ ]:
"""Readable analysis code inserted into the public replication notebook."""
from pathlib import Path
import json
import warnings
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

BASIC = ['number_of_issues', 'number_of_parties', 'number_of_compatible_issues',
         'automated_counterpart', 'non_unanimous_rule']
CONTINUOUS = ['search_complexity', 'bargaining_zone_share', *BASIC[:3]]
MODELS = {
    'M1': ['search_complexity'],
    'M2': ['search_complexity', *BASIC],
    'M3': ['search_complexity', 'bargaining_zone_share', *BASIC],
    'M4': ['bargaining_zone_share'],
    'M5': ['bargaining_zone_share', *BASIC],
    'M6': ['search_complexity', 'bargaining_zone_share', *BASIC],
}
TESTS = {'H1':('M4','bargaining_zone_share',1), 'H2':('M1','search_complexity',-1),
         'H4':('M3','search_complexity',-1)}

def analysis_sample(data, tier):
    """Use the strict outcome evidence only for rows in the strict analysis."""
    sample = data.loc[data[f'{tier}_eligible']].copy()
    if tier == 'strict':
        for col in ['agreement_rate','agreements','negotiations']:
            sample[col] = sample[f'strict_{col}']
    return sample.reset_index(drop=True)

def validate_data(data):
    """Check units, identifiers and mathematical limits before analysis."""
    assert data.observation_id.is_unique
    assert not (data.strict_eligible & ~data.main_eligible).any()
    for col in ['agreement_rate','bargaining_zone_share','search_complexity']:
        assert data[col].between(0,1).all(), col
    assert (data.search_complexity <= 1-data.bargaining_zone_share+1e-9).all()
    assert (data.number_of_issues >= 2).all()
    assert (data.number_of_parties >= 2).all()
    assert data.loc[data.main_eligible, ['agreement_rate','search_complexity','bargaining_zone_share',*BASIC]].notna().all().all()
    assert data.loc[data.strict_eligible,'strict_agreement_rate'].between(0,1).all()
    # A mean of study rates can differ from the pooled numerator / denominator.
    # Therefore no check incorrectly forces every primary rate to equal A / N.
    for tier in ['main','strict']:
        sample = analysis_sample(data,tier)
        complete = sample[['agreements','negotiations']].dropna()
        assert (complete.agreements >= 0).all()
        assert (complete.agreements <= complete.negotiations).all()
    return {'retained_records':len(data), **{tier:{'observations':len(analysis_sample(data,tier)),
            'scenario_names':analysis_sample(data,tier).negotiation_scenario.nunique()} for tier in ['main','strict']}}

def fit(sample, model):
    """Estimate OLS, or negotiation-count WLS for M6, with HC3 uncertainty.

    We standardize continuous X variables *after* choosing the model's rows.
    Y stays a proportion. A coefficient of -.10 means 10 percentage points less
    agreement for a one-SD increase in X. These are not fully standardized betas.
    """
    predictors = MODELS[model]
    use = sample.dropna(subset=['agreement_rate',*predictors]).copy()
    if model == 'M6':
        use = use.loc[use.negotiations.notna() & use.negotiations.gt(0)]
    X = use[predictors].astype(float).copy()
    for col in predictors:
        if col in CONTINUOUS:
            sd = X[col].std(ddof=0)
            X[col] = (X[col]-X[col].mean())/sd if sd > 1e-12 else 0.
    X.insert(0,'const',1.)
    # Rare controls can be constant or redundant in a resample. Keep the focal
    # predictors first, then retain only controls adding independent information.
    kept, dropped = ['const'], []
    for col in predictors:
        if np.linalg.matrix_rank(X[[*kept,col]].to_numpy()) > len(kept):
            kept.append(col)
        elif col in ['search_complexity','bargaining_zone_share']:
            raise ValueError(f'{model}: focal predictor {col} is not identifiable')
        else:
            dropped.append(col)
    X = X[kept]
    regression = sm.WLS(use.agreement_rate,X,weights=use.negotiations) if model=='M6' else sm.OLS(use.agreement_rate,X)
    ordinary = regression.fit()
    # A binary control occurring in only one row fits that row exactly. HC3's
    # residual/(1-leverage) then becomes 0/0. The other slopes are exactly the
    # same after absorbing that dummy and its singleton row. Compute their HC3
    # covariance on the remaining rows, preserving the original X scaling and
    # degrees of freedom. Uncertainty for the singleton dummy is unavailable.
    singleton = [col for col in X.columns if col!='const' and
                 col not in ['search_complexity','bargaining_zone_share'] and
                 int(X[col].ne(0).sum())==1]
    if singleton:
        keep_rows = ~X[singleton].ne(0).any(axis=1)
        remaining = [col for col in X if col not in singleton]
        reduced = sm.WLS(use.loc[keep_rows,'agreement_rate'],X.loc[keep_rows,remaining],
                         weights=use.loc[keep_rows,'negotiations']) if model=='M6' else sm.OLS(
                         use.loc[keep_rows,'agreement_rate'],X.loc[keep_rows,remaining])
        reduced_fit = reduced.fit(cov_type='HC3',use_t=True)
        np.testing.assert_allclose(ordinary.params[remaining],reduced_fit.params,atol=1e-9)
        se = reduced_fit.bse.reindex(X.columns)
        pv = pd.Series(2*stats.t.sf(np.abs(ordinary.params/se),ordinary.df_resid),index=X.columns)
        critical = stats.t.ppf(.975,ordinary.df_resid)
        ci = pd.DataFrame({0:ordinary.params-critical*se,1:ordinary.params+critical*se})
        fitted = SimpleNamespace(params=ordinary.params,bse=se,pvalues=pv,conf_int=lambda:ci,
                                 nobs=ordinary.nobs,rsquared=ordinary.rsquared,
                                 rsquared_adj=ordinary.rsquared_adj,fittedvalues=ordinary.fittedvalues)
        dropped.extend(f'{col}: singleton dummy; coefficient retained, HC3 SE unavailable' for col in singleton)
    else:
        fitted = regression.fit(cov_type='HC3',use_t=True)
    focal = [col for col in X if col in ['search_complexity','bargaining_zone_share']]
    if not np.isfinite(fitted.bse[focal]).all():
        raise ValueError(f'{model}: HC3 inference for a focal predictor is not finite')
    terms = pd.DataFrame({'term':fitted.params.index,'b':fitted.params.values,
                          'se_hc3':fitted.bse.values,'p':fitted.pvalues.values,
                          'ci_low':fitted.conf_int().iloc[:,0].values,
                          'ci_high':fitted.conf_int().iloc[:,1].values})
    return fitted, terms, use, dropped

def all_models(sample, tier):
    coefficients, models = [], []
    for model in MODELS:
        fitted, terms, use, dropped = fit(sample,model)
        terms.insert(0,'model',model)
        terms.insert(0,'dataset',tier)
        coefficients.append(terms)
        models.append({'dataset':tier,'model':model,'n':len(use),
                       'scenario_names':use.negotiation_scenario.nunique(),
                       'r_squared':float(fitted.rsquared),
                       'adjusted_r_squared':float(fitted.rsquared_adj),
                       'dropped_controls':'; '.join(dropped),
                       'predictions_below_zero':int((fitted.fittedvalues<0).sum()),
                       'predictions_above_one':int((fitted.fittedvalues>1).sum())})
    return pd.concat(coefficients,ignore_index=True),pd.DataFrame(models)

def hypothesis_estimates(sample):
    out = {}
    for h,(model,term,direction) in TESTS.items():
        fitted,_,_,_ = fit(sample,model)
        out[h] = {'estimate':float(fitted.params[term]),'p':float(fitted.pvalues[term]),
                  'full_sample_pass':bool(direction*fitted.params[term]>0 and fitted.pvalues[term]<.05)}
    m2 = fit(sample,'M2')[0]
    m5 = fit(sample,'M5')[0]
    assert m2.nobs == m5.nobs
    delta = m2.rsquared-m5.rsquared
    out['H3'] = {'estimate':float(delta),'p':None,'full_sample_pass':bool(delta>0)}
    return out

def fast_estimates(sample, weighted=False):
    """Least squares point estimates for the resampling calculations.

    This is algebraically the same model as statsmodels. Robust standard errors
    are unnecessary inside the percentile bootstrap, which uses only estimates.
    """
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    y = sample.agreement_rate.to_numpy(float)
    raw = sample[['search_complexity','bargaining_zone_share',*BASIC]].to_numpy(float)
    X = raw.copy()
    for j in range(5):
        sd = raw[:,j].std(ddof=0)
        X[:,j] = (raw[:,j]-raw[:,j].mean())/sd if sd>1e-12 else 0.
    X = np.column_stack([np.ones(len(y)),X])
    weights = sample.negotiations.to_numpy(float) if weighted else np.ones(len(y))
    root = np.sqrt(weights)
    total = np.sum(weights*(y-np.average(y,weights=weights))**2)
    if total<=0:
        raise ValueError('No outcome variation in resample')
    indices = {'M1':[0,1],'M2':[0,1,3,4,5,6,7], 'M3':list(range(8)),
               'M4':[0,2],'M5':[0,2,3,4,5,6,7]}
    fitted = {}
    for model in (['M3'] if weighted else indices):
        cols = indices[model]
        matrix = X[:,cols]
        # Reduce redundant controls in the same fixed order as fit().
        keep = [0]
        for j in range(1,len(cols)):
            if np.linalg.matrix_rank(matrix[:,[*keep,j]]) > len(keep):
                keep.append(j)
            elif cols[j] in [1,2]:
                raise ValueError('Focal predictor not identified in resample')
        matrix = matrix[:,keep]
        coefficient = np.linalg.lstsq(matrix*root[:,None],y*root,rcond=None)[0]
        r2 = 1-np.sum(weights*(y-matrix@coefficient)**2)/total
        fitted[model] = (coefficient,r2)
    if weighted:
        return {'M6':float(fitted['M3'][0][1])}
    return {'H1':float(fitted['M4'][0][1]),'H2':float(fitted['M1'][0][1]),
            'H3':float(fitted['M2'][1]-fitted['M5'][1]),'H4':float(fitted['M3'][0][1])}

def bootstrap(sample, draws=5000, seed=20260921, whole_scenarios=False, weighted=False):
    """Resample observations, or scenario names for the supplementary check."""
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    rng = np.random.default_rng(seed)
    groups = [g for _,g in sample.groupby('negotiation_scenario',sort=False)]
    values, failures = [], []
    for draw in range(draws):
        if whole_scenarios:
            resample = pd.concat([groups[j] for j in rng.integers(0,len(groups),len(groups))],ignore_index=True)
        else:
            resample = sample.iloc[rng.integers(0,len(sample),len(sample))]
        try:
            values.append(fast_estimates(resample,weighted=weighted))
        except ValueError as error:
            failures.append({'draw':draw+1,'reason':str(error)})
    results = pd.DataFrame(values)
    summary = []
    for h in results:
        low,high = results[h].quantile([.025,.975])
        positive = h in ['H1','H3']
        summary.append({'test':h,'ci_low':low,'ci_high':high,
                        'draws_requested':draws,'draws_completed':len(results),
                        'invalid_draws':len(failures),
                        'passes_interval_rule':bool(low>0 if positive else high<0)})
    return pd.DataFrame(summary),pd.DataFrame(failures)

def omission_checks(sample, whole_scenarios=False, weighted=False):
    """Apply direction AND significance, as required in the preregistration."""
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    key = 'negotiation_scenario' if whole_scenarios else 'observation_id'
    rows = []
    for omitted in sample[key].unique():
        use = sample.loc[sample[key].ne(omitted)]
        if weighted:
            fitted = fit(use,'M6')[0]
            rows.append({'omitted':omitted,'test':'M6','estimate':float(fitted.params['search_complexity']),
                         'p':float(fitted.pvalues['search_complexity']),
                         'passes':bool(fitted.params['search_complexity']<0 and fitted.pvalues['search_complexity']<.05)})
        else:
            for h,value in hypothesis_estimates(use).items():
                rows.append({'omitted':omitted,'test':h,'estimate':value['estimate'],
                             'p':value['p'],'passes':value['full_sample_pass']})
    return pd.DataFrame(rows)

def collapse_scenarios(sample):
    """Average structural versions into one row per named scenario, as specified."""
    return sample.groupby('negotiation_scenario',as_index=False)[
        ['agreement_rate','search_complexity','bargaining_zone_share',*BASIC]].mean()

def describe(sample):
    columns = ['agreement_rate','bargaining_zone_share','search_complexity',*BASIC]
    summary = sample[columns].agg(['count','mean','std','min','max']).T
    correlation = sample[columns].corr()
    p = pd.DataFrame(np.nan,index=columns,columns=columns)
    for left in columns:
        for right in columns:
            if sample[left].std()>0 and sample[right].std()>0:
                p.loc[left,right] = stats.pearsonr(sample[left],sample[right]).pvalue
    return summary,correlation,p

def run_all(data, output, draws=5000):
    output = Path(output)
    output.mkdir(parents=True,exist_ok=True)
    counts = validate_data(data)
    coefficients,models,bootstrap_rows,omission_rows,collapsed_rows,support = [],[],[],[],[],[]
    for offset,tier in enumerate(['main','strict']):
        sample = analysis_sample(data,tier)
        print(f'{tier}: {len(sample)} observations; {draws} draws for each specified resampling check',flush=True)
        co,mo = all_models(sample,tier)
        coefficients.append(co); models.append(mo)
        d,c,p = describe(sample)
        d.to_csv(output/f'{tier}_descriptive_statistics.csv')
        c.to_csv(output/f'{tier}_correlations.csv')
        p.to_csv(output/f'{tier}_correlation_p_values.csv')
        primary = hypothesis_estimates(sample)
        for whole in [False,True]:
            for weighted in [False,True]:
                label = ('scenario' if whole else 'observation') + ('_weighted' if weighted else '')
                intervals,failures = bootstrap(sample,draws,20260921+offset+int(whole)*100+int(weighted)*200,whole,weighted)
                intervals.insert(0,'resampling',label); intervals.insert(0,'dataset',tier)
                bootstrap_rows.append(intervals)
                if len(failures):
                    failures.to_csv(output/f'{tier}_{label}_invalid_draws.csv',index=False)
                omissions = omission_checks(sample,whole,weighted)
                omissions.insert(0,'omission',label); omissions.insert(0,'dataset',tier)
                omission_rows.append(omissions)
                if not whole and not weighted:
                    for h,full in primary.items():
                        bs = intervals.loc[intervals.test.eq(h)].iloc[0]
                        omit = omissions.loc[omissions.test.eq(h)]
                        support.append({'dataset':tier,'hypothesis':h,**full,
                            'bootstrap_ci_low':bs.ci_low,'bootstrap_ci_high':bs.ci_high,
                            'bootstrap_pass':bool(bs.passes_interval_rule),
                            'omission_refits':len(omit),'omission_refits_passing':int(omit.passes.sum()),
                            'supported_by_specified_tests':bool(full['full_sample_pass'] and bs.passes_interval_rule and omit.passes.all())})
                print(f'  completed {label}',flush=True)
        collapsed = collapse_scenarios(sample)
        co,mo = all_models_for_collapsed(collapsed,tier)
        collapsed_rows.append(co.merge(mo,on=['dataset','model']))
    for name,items in [('coefficients',coefficients),('model_fit',models),('bootstrap_intervals',bootstrap_rows),
                       ('omission_checks',omission_rows),('scenario_averaged_results',collapsed_rows)]:
        pd.concat(items,ignore_index=True).to_csv(output/f'{name}.csv',index=False)
    pd.DataFrame(support).sort_values(['dataset','hypothesis']).to_csv(output/'hypothesis_tests.csv',index=False)
    (output/'counts.json').write_text(json.dumps(counts,indent=2),encoding='utf-8')
    return pd.DataFrame(support).sort_values(['dataset','hypothesis'])

def all_models_for_collapsed(sample,tier):
    co,mo = [],[]
    for model in ['M1','M2','M3','M4','M5']:
        fitted,terms,use,dropped = fit(sample,model)
        terms.insert(0,'model',model); terms.insert(0,'dataset',tier)
        co.append(terms)
        mo.append({'dataset':tier,'model':model,'n':len(use),'r_squared':float(fitted.rsquared)})
    return pd.concat(co,ignore_index=True),pd.DataFrame(mo)


In [ ]:
counts = validate_data(data)
print(json.dumps(counts,indent=2))
for tier in ['main','strict']:
    sample = analysis_sample(data,tier)
    estimates = hypothesis_estimates(sample)
    for h,value in fast_estimates(sample).items():
        np.testing.assert_allclose(value,estimates[h]['estimate'],atol=1e-10)
    np.testing.assert_allclose(fast_estimates(sample,weighted=True)['M6'],
                              fit(sample,'M6')[0].params['search_complexity'],atol=1e-10)
print('Point-estimate checks passed.')

## Fit the models and run every specified check

For H1, H2 and H4, the specified support rule requires a significant coefficient
in the expected direction, a 95% observation-bootstrap interval entirely in that
direction, and a significant coefficient in every leave-one-observation-out fit.
For H3, the paired R² difference and its bootstrap interval must be positive,
and the difference must remain positive after every omission.

Bootstrapping samples rows with replacement, then refits both competing models
on the same resampled rows. Five thousand draws place about 125 draws below each
2.5th-percentile endpoint. They improve numerical stability; they do not create
new evidence or increase the number of independent negotiations.

Supplementary checks repeat the analysis in strict, average versions sharing a
scenario name, resample whole scenario names, omit one whole scenario at a time,
and repeat weighted Model 6 with its original row weights. Every omission result
is saved, including failures. Invalid resamples, if any, are counted and logged.
These checks do not replace the primary support rule.

In [ ]:
support = run_all(data,OUTPUT,draws=DRAWS)
support

## Tables and interpretation

The model table separates coefficients, robust standard errors and p values.
The dependent variable is agreement rate. R² describes in-sample fit, not
out-of-sample prediction. Models 2 and 5 use the same observations and controls;
their R² difference is a paired comparison of non-nested models, not a nested F test.

In the audited data, the main sample meets the specified tests for all four
hypotheses. The stricter sample meets H2-H4 but not H1. Some supplementary
intervals include zero, so the results should not be described as passing every
robustness test. An interval crossing zero means the check does not distinguish
the expected effect from zero at that interval level.

In [ ]:
models = pd.read_csv(OUTPUT/'model_fit.csv')
coefficients = pd.read_csv(OUTPUT/'coefficients.csv')
focal = coefficients.loc[coefficients.term.isin(['search_complexity','bargaining_zone_share'])]
focal.merge(models[['dataset','model','n','r_squared','adjusted_r_squared']],on=['dataset','model'])

In [ ]:
pd.read_csv(OUTPUT/'bootstrap_intervals.csv')

In [ ]:
omissions = pd.read_csv(OUTPUT/'omission_checks.csv')
omissions.groupby(['dataset','omission','test']).agg(
    refits=('passes','size'),refits_passing=('passes','sum'),largest_p=('p','max'))

In [ ]:
pd.read_csv(OUTPUT/'scenario_averaged_results.csv').loc[
    lambda frame:frame.term.isin(['search_complexity','bargaining_zone_share'])]

## Figures

Figure 1 shows the raw associations, including overlap at 100% agreement.
Figure 2 shows coefficients and HC3 intervals across specifications. Each point
is an association. Neither figure establishes a causal effect.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams.update({'font.family':'Times New Roman','font.size':14,
                     'axes.labelsize':16,'axes.titlesize':16,'xtick.labelsize':13,
                     'ytick.labelsize':13,'svg.fonttype':'none'})
sample = analysis_sample(data,'main')
fig,axes = plt.subplots(1,2,figsize=(11,4.5),constrained_layout=True)
for ax,x,label in zip(axes,['bargaining_zone_share','search_complexity'],
                      ['Bargaining-zone share','Search complexity']):
    ax.scatter(sample[x],sample.agreement_rate,color='black',alpha=.45,s=36,
               edgecolors='white',linewidths=.4)
    ax.set(xlabel=label,ylabel='Observed agreement rate',xlim=(-.03,1.03),ylim=(-.03,1.03))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(False)
fig.suptitle('Structural features and observed agreement rates',fontsize=18)
fig.savefig(OUTPUT/'figure_1_associations.png',dpi=300,bbox_inches='tight')
fig.savefig(OUTPUT/'figure_1_associations.svg',bbox_inches='tight')
plt.close(fig)

co = pd.read_csv(OUTPUT/'coefficients.csv')
fig,ax = plt.subplots(figsize=(9,5.4),constrained_layout=True)
positions = {'M1':3,'M2':2,'M3':1,'M6':0}
for tier,offset,color,marker in [('main',.10,'black','o'),('strict',-.10,'#777777','s')]:
    for model,y in positions.items():
        row = co.loc[co.dataset.eq(tier)&co.model.eq(model)&co.term.eq('search_complexity')].iloc[0]
        ax.errorbar(row.b*100,y+offset,xerr=[[(row.b-row.ci_low)*100],[(row.ci_high-row.b)*100]],
                    fmt=marker,color=color,capsize=4,markersize=7,
                    label=tier.capitalize() if model=='M1' else None)
ax.axvline(0,color='#aaaaaa',linewidth=1)
ax.set_yticks([3,2,1,0],['M1  Search complexity','M2  + structural controls',
                        'M3  + bargaining-zone share','M6  Weighted by negotiations'])
ax.set_xlabel('Agreement-rate difference per one-SD increase (percentage points)')
ax.set_title('Search complexity across model specifications',pad=48)
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1.015),ncol=2)
ax.spines[['top','right','left']].set_visible(False)
ax.grid(False)
fig.savefig(OUTPUT/'figure_2_coefficients.png',dpi=300,bbox_inches='tight')
fig.savefig(OUTPUT/'figure_2_coefficients.svg',bbox_inches='tight')
plt.close(fig)
print('Saved both figures as PNG and SVG files in outputs/.')

## References and reproducibility notes

- MacKinnon, J. G., & White, H. (1985). Some heteroskedasticity-consistent covariance matrix estimators with improved finite sample properties. *Journal of Econometrics, 29*, 305–325. https://doi.org/10.1016/0304-4076(85)90158-7
- Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall. https://doi.org/10.1007/978-1-4899-4541-9
- Warsitzka, M., Zhang, H., Beersma, B., Freund, P. A., & Trötschel, R. (2024). Expanding the pie or spoiling the cake? How the number of negotiation issues affects integrative bargaining. *Journal of Applied Psychology, 109*, 1224–1249. https://doi.org/10.1037/apl0001149

Source coding follows the archived OSF plan. The repository still needs the
permanent public registration link and a verified registration date before
publication. The notebook does not certify the chronology of prior analyses.
Study and structural sources are in the two evidence CSV files.